In [6]:
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer,DataCollatorForTokenClassification, AutoModel
import numpy as np
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
from torch.optim import AdamW


train_path = "project/en_ewt-ud-dev.iob2"
dev_path =  "project/en_ewt-ud-dev.iob2"
test_path = "project/en_ewt-ud-test-masked.iob2"

Just run the cells as they are and it should compute. The train path is used for training, dev path is for evaluating and test is for predicting the final values.

In [ ]:
def read_ewt_iob2(file_path):
    sentences = []
    current_tokens = []
    current_tags = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith("#"):
                continue
            if not line:
                if current_tokens:
                    sentences.append({"tokens": current_tokens, "tags": current_tags})
                    current_tokens, current_tags = [], []
                continue
            parts = line.split()
            if len(parts) >= 3:
                current_tokens.append(parts[1])
                current_tags.append(parts[2])
        if current_tokens:
            sentences.append({"tokens": current_tokens, "tags": current_tags})
                
    return sentences

train_data = read_ewt_iob2(train_path)
dev_data = read_ewt_iob2(dev_path)

Loaded 2001 sentences.
Example: {'tokens': ['where', 'can', 'I', 'get', 'morcillas', 'in', 'tampa', 'bay', ',', 'I', 'will', 'like', 'the', 'argentinian', 'type', ',', 'but', 'I', 'will', 'to', 'try', 'anothers', 'please', '?'], 'tags': ['O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [8]:
unique_tags = sorted(list(set(tag for s in train_data for tag in s["tags"])))
tag2id = {tag: i for i, tag in enumerate(unique_tags)}
id2tag = {i: tag for tag, i in tag2id.items()}

In [ ]:
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext" #Our intended model to use.
tokenizer = AutoTokenizer.from_pretrained(model_name) #Borrows it tokenizer

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  
            elif word_idx != previous_word_idx:
                label_ids.append(tag2id[label[word_idx]]) 
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [10]:
train_dataset = Dataset.from_list(train_data)
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)

dev_dataset = Dataset.from_list(dev_data)
tokenized_dev = dev_dataset.map(tokenize_and_align_labels, batched=True, load_from_cache_file=False)

Map: 100%|██████████| 2001/2001 [00:00<00:00, 35862.85 examples/s]


In [11]:
data_collator = DataCollatorForTokenClassification(tokenizer)


train_dataloader = DataLoader(tokenized_train,
                              shuffle=True,
                              batch_size = 16,
                              collate_fn=data_collator)
dev_dataloader = DataLoader(tokenized_dev,
                            batch_size=16,
                            collate_fn=data_collator)





In [12]:
class NER(nn.Module):
    def __init__(self, model_name, num_labels):
        super(NER,self).__init__()
        self.bert = AutoModel.from_pretrained(model_name) # initialize weights from pretrained.
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_output = outputs.last_hidden_state
        seq_output = self.dropout(seq_output)
        logits = self.classifier(seq_output)
    
        loss = None
        if labels is not None:
            loss_fun = nn.CrossEntropyLoss()
            active_loss = attention_mask.view(-1) == 1
            active_logits = logits.view(-1, self.classifier.out_features)
            active_labels = torch.where(
                active_loss, labels.view(-1), torch.tensor(loss_fun.ignore_index).type_as(labels)
            )
            loss = loss_fun(active_logits, active_labels)

        return (loss, logits) if loss is not None else logits

In [13]:
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_dev.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model = NER(model_name, num_labels=len(unique_tags)).to(device)
optimizer = AdamW(model.parameters(), lr=1e-5)

model.train()
for epoch in range(3):
    for batch in train_dataloader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, logits = model(**batch)
        loss.backward()
        optimizer.step()
    #Just to see if the model is learning.
    print(f"Epoch {epoch} Loss: {loss.item()}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 63944.42it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 0 complete. Loss: 0.011114458553493023
Epoch 1 complete. Loss: 0.09669161587953568
Epoch 2 complete. Loss: 0.00313826953060925


In [15]:
def predict(model, dataloader, device, tags):
    model.eval()
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            logits = model(input_ids,attention_mask)
            preds = torch.argmax(logits,dim=-1).cpu().numpy()
            label_ids = labels.cpu().numpy()
            for i in range(len(preds)):
                sen_pred = [id2tag[p] for p, l in zip(preds[i],label_ids[i]) if l != -100]
                all_predictions.append(sen_pred)
    return all_predictions

In [16]:
test_raw = read_ewt_iob2(test_path)
test_dataset = Dataset.from_list(test_raw)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataloader = DataLoader(
    tokenized_test,
    batch_size=16,
    collate_fn=data_collator
)

Map: 100%|██████████| 2077/2077 [00:00<00:00, 28846.45 examples/s]


In [18]:
def save_formatted_predictions(original_data, predictions, output_file="baseline_results_dev.txt"):
    with open(output_file, 'w', encoding='utf-8') as f:

        for i, sentence in enumerate(original_data):
            tokens = sentence["tokens"]
            preds = predictions[i]
        
            for idx, (token, tag) in enumerate(zip(tokens, preds), 1):

                line = f"{idx}\t{token}\t{tag}\t-\t-\n"
                f.write(line)
            f.write("\n")

predictions = predict(model, test_dataloader, device, id2tag)
save_formatted_predictions(test_raw, predictions)